# Accessing Resources

Resources let the server expose information that gets **included directly in prompts**, instead of requiring a tool call to fetch data — a more efficient way to give the model context.

Flow: a user types *"What's in the @..."* → your code recognizes a resource request → sends a **`ReadResourceRequest`** to the server → gets back a **`ReadResourceResult`** with the content.

> Implement this in **`cli-project/mcp_client.py`** (the `read_resource` `TODO`). `skip-worktree` keeps your edits local; answer key in `cli-project-complete/`.

## Implementing `read_resource`

Add the imports:

```python
import json
from pydantic import AnyUrl
```

The function requests the resource and processes the response by MIME type:

```python
async def read_resource(self, uri: str) -> Any:
    result = await self.session().read_resource(AnyUrl(uri))
    resource = result.contents[0]

    if isinstance(resource, types.TextResourceContents):
        if resource.mimeType == "application/json":
            return json.loads(resource.text)

    return resource.text
```

## The response structure

The server returns a result with a **`contents` list**. Take the first element (`contents[0]`) — you typically want one resource at a time. Each entry carries:

- the actual **content** (text or data),
- a **MIME type** telling you how to parse it,
- other metadata.

## Content-type handling

The function branches on MIME type:

- **`application/json`** → parse the text with `json.loads` and return the parsed object.
- **otherwise** → return the raw `resource.text`.

That handles structured data and plain text with the same function. Note this mirrors the server's `mime_type` hint from the previous lesson — the server labels it, the client parses accordingly.

## Testing resource access

Test through the CLI. Type `@` followed by a resource name and the app will:

1. Show available resources in an **autocomplete** list.
2. Let you select with arrow keys + space.
3. Include the resource content **directly in your prompt**.
4. Send everything to the model **without extra tool calls**.

Smoother than having the model make separate tool calls — the content is part of the initial context, so answers about the data are immediate.

> This is why resources exist alongside tools: read-only context that's cheaper and lower-latency than a tool round-trip.

Next: **Defining prompts** — expose reusable, server-defined prompts (the `/command` feature).